In [1]:
!pip install remfile dandi pynwb h5py scikit-learn matplotlib numpy

In [ ]:
import numpy as np
import remfile
import h5py
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import f_classif
from scipy.ndimage import gaussian_filter1d
import time
import warnings

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION 
# ============================================================================

DANDISET_ID = "000402"
TARGET_STIMULI = ['Cinematic', 'Rendered', 'sports1m']
INTERVAL_NAME = 'Clip'

# Feature extraction settings
N_TOP_NEURONS_PER_PLANE = 250  # Increased from 200
USE_TEMPORAL_BINS = True  # Extract features from multiple time windows
N_TIME_BINS = 3  # Early, middle, late response
SMOOTH_TRACES = True  # Apply gaussian smoothing to reduce noise
SMOOTH_SIGMA = 2  # Smoothing parameter

# Model settings
N_FOLDS = 5
MAX_FILES = 3  # Set to None for all files

# ============================================================================
# DATA LOADING
# ============================================================================

def get_all_nwb_assets(dandiset_id="000402"):
    """Get all NWB file assets from DANDI."""
    client = DandiAPIClient()
    dandiset = client.get_dandiset(dandiset_id, "draft")
    all_assets = list(dandiset.get_assets())
    nwb_assets = [a for a in all_assets if a.path.endswith('.nwb')]
    return nwb_assets


def load_nwb_file(asset):
    """Load an NWB file from a DANDI asset."""
    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)
    rf = remfile.File(s3_url)
    h5 = h5py.File(rf, "r")
    io = NWBHDF5IO(file=h5, load_namespaces=True)
    nwb = io.read()
    return nwb


def get_stimulus_timing(nwb, interval_name='Clip'):
    """Extract stimulus timing and labels."""
    interval = nwb.intervals[interval_name]
    starts = np.array(interval.start_time[:])
    stops = np.array(interval.stop_time[:])
    labels = np.array(interval.short_movie_name[:])
    return starts, stops, labels


def get_all_roi_series(nwb):
    """Get all ROI response series."""
    ophys = nwb.processing['ophys']
    fluorescence = ophys.data_interfaces['Fluorescence']
    all_series = fluorescence.roi_response_series
    return [(name, all_series[name]) for name in sorted(all_series.keys())]


# ============================================================================
# IMPROVED FEATURE EXTRACTION
# ============================================================================

def extract_temporal_features(neural_data, n_bins=3):
    """
    Extract features from multiple temporal bins.
    Instead of just mean, get mean from early/middle/late parts of response.
    
    Args:
        neural_data: (n_timepoints, n_neurons)
        n_bins: Number of time bins to split into
    
    Returns:
        features: (n_neurons * n_bins,) flattened features
    """
    n_timepoints, n_neurons = neural_data.shape
    bin_size = n_timepoints // n_bins
    
    features = []
    for bin_idx in range(n_bins):
        start = bin_idx * bin_size
        end = start + bin_size if bin_idx < n_bins - 1 else n_timepoints
        bin_mean = np.mean(neural_data[start:end, :], axis=0)
        features.append(bin_mean)
    
    # Flatten: [early_neuron1, early_neuron2, ..., middle_neuron1, ...]
    return np.concatenate(features)


def select_best_neurons_single_plane(rs, timestamps, starts, stops, labels, 
                                     target_stimuli, n_select=250):
    """Select most informative neurons from a single plane."""
    all_neurons = list(range(rs.data.shape[1]))
    
    # Use simple features for selection (faster)
    X_all, y_all = extract_features_simple(
        rs, timestamps, starts, stops, labels,
        all_neurons, target_stimuli
    )
    
    f_scores, _ = f_classif(X_all, y_all)
    best_indices = np.argsort(f_scores)[-n_select:]
    best_indices = sorted(best_indices.tolist())
    
    return best_indices


def extract_features_simple(rs, timestamps, starts, stops, labels,
                            neuron_indices, target_stimuli):
    """Simple mean features for neuron selection."""
    stim_to_idx = {stim: i for i, stim in enumerate(target_stimuli)}
    
    trials = []
    trial_labels = []
    
    for i in range(len(starts)):
        if labels[i] not in target_stimuli:
            continue
        
        start_idx = np.searchsorted(timestamps, starts[i])
        stop_idx = np.searchsorted(timestamps, stops[i])
        
        neural_data = np.array(rs.data[start_idx:stop_idx, neuron_indices])
        mean_activity = np.mean(neural_data, axis=0)
        
        trials.append(mean_activity)
        trial_labels.append(stim_to_idx[labels[i]])
    
    return np.array(trials), np.array(trial_labels)


def extract_features_advanced(rs, timestamps, starts, stops, labels,
                              neuron_indices, target_stimuli, 
                              use_temporal_bins=True, n_bins=3, 
                              smooth=True, sigma=2):
    """
    Advanced feature extraction with temporal binning and smoothing.
    
    Returns richer features that should improve accuracy.
    """
    stim_to_idx = {stim: i for i, stim in enumerate(target_stimuli)}
    
    trials = []
    trial_labels = []
    
    for i in range(len(starts)):
        if labels[i] not in target_stimuli:
            continue
        
        start_idx = np.searchsorted(timestamps, starts[i])
        stop_idx = np.searchsorted(timestamps, stops[i])
        
        neural_data = np.array(rs.data[start_idx:stop_idx, neuron_indices])
        
        # Apply smoothing if enabled
        if smooth and neural_data.shape[0] > sigma * 3:
            neural_data = gaussian_filter1d(neural_data, sigma=sigma, axis=0)
        
        # Extract features
        if use_temporal_bins:
            features = extract_temporal_features(neural_data, n_bins=n_bins)
        else:
            features = np.mean(neural_data, axis=0)
        
        trials.append(features)
        trial_labels.append(stim_to_idx[labels[i]])
    
    return np.array(trials), np.array(trial_labels)


# ============================================================================
# MULTI-PLANE FEATURE EXTRACTION
# ============================================================================

def select_neurons_all_planes(nwb, starts, stops, labels, 
                              target_stimuli, n_select_per_plane=250):
    """Select best neurons from all planes."""
    roi_series_list = get_all_roi_series(nwb)
    
    selected_neurons = {}
    
    print(f"  Selecting top {n_select_per_plane} neurons per plane...")
    
    for plane_name, rs in roi_series_list:
        timestamps = np.array(rs.timestamps[:])
        n_neurons = rs.data.shape[1]
        n_select = min(n_select_per_plane, n_neurons)
        
        neuron_indices = select_best_neurons_single_plane(
            rs, timestamps, starts, stops, labels,
            target_stimuli, n_select=n_select
        )
        
        selected_neurons[plane_name] = neuron_indices
        print(f"    {plane_name}: {len(neuron_indices)}/{n_neurons} neurons")
    
    return selected_neurons


def extract_features_all_planes(nwb, starts, stops, labels,
                                target_stimuli, selected_neurons):
    """Extract and concatenate features from all planes."""
    roi_series_list = get_all_roi_series(nwb)
    
    plane_features = {}
    
    for plane_name, rs in roi_series_list:
        if plane_name not in selected_neurons:
            continue
        
        timestamps = np.array(rs.timestamps[:])
        neuron_indices = selected_neurons[plane_name]
        
        X_plane, y_plane = extract_features_advanced(
            rs, timestamps, starts, stops, labels,
            neuron_indices, target_stimuli,
            use_temporal_bins=USE_TEMPORAL_BINS,
            n_bins=N_TIME_BINS,
            smooth=SMOOTH_TRACES,
            sigma=SMOOTH_SIGMA
        )
        
        plane_features[plane_name] = X_plane
    
    # Concatenate all planes
    X_concat = np.hstack([plane_features[name] for name in sorted(plane_features.keys())])
    y = y_plane
    
    return X_concat, y


# ============================================================================
# MODEL TRAINING - OPTIMIZED LOGISTIC REGRESSION
# ============================================================================

def train_optimized_logistic(X, y, n_folds=5):
    """Train optimized logistic regression."""
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(random_state=42))
    ])
    
    # Simplified param grid (removed incompatible combinations)
    param_grid = [
        {
            'classifier__C': [0.001, 0.01, 0.1, 1.0, 10, 100],
            'classifier__penalty': ['l2'],
            'classifier__solver': ['lbfgs', 'saga'],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__max_iter': [5000]
        },
        {
            'classifier__C': [0.001, 0.01, 0.1, 1.0, 10, 100],
            'classifier__penalty': ['l1'],
            'classifier__solver': ['saga'],  # Only saga supports l1
            'classifier__class_weight': [None, 'balanced'],
            'classifier__max_iter': [5000]
        }
    ]
    
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        verbose=0
    )
    
    grid_search.fit(X, y)
    return grid_search.best_estimator_, grid_search.best_params_


def evaluate_model(pipeline, X, y, n_folds=5):
    """Evaluate model with cross-validation."""
    from sklearn.base import clone
    
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_accuracies = []
    
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        model = clone(pipeline)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        accuracy = np.mean(y_test == y_pred)
        fold_accuracies.append(accuracy)
    
    return fold_accuracies


# ============================================================================
# MAIN ANALYSIS
# ============================================================================

def analyze_file(nwb, file_name):
    """Analyze a single file with optimized multi-plane decoder."""
    print(f"\n{'='*70}")
    print(f"File: {file_name.split('/')[-1]}")
    print(f"{'='*70}")
    
    start_time = time.time()
    
    # Get stimulus info
    starts, stops, labels = get_stimulus_timing(nwb, INTERVAL_NAME)
    
    # Select neurons from all planes
    selected_neurons = select_neurons_all_planes(
        nwb, starts, stops, labels, TARGET_STIMULI, N_TOP_NEURONS_PER_PLANE
    )
    
    # Extract features
    if USE_TEMPORAL_BINS:
        print(f"  Using temporal binning: {N_TIME_BINS} bins per trial")
    if SMOOTH_TRACES:
        print(f"  Smoothing traces: sigma={SMOOTH_SIGMA}")
    
    X, y = extract_features_all_planes(
        nwb, starts, stops, labels, TARGET_STIMULI, selected_neurons
    )
    
    total_neurons = sum(len(indices) for indices in selected_neurons.values())
    print(f"\nDataset shape:")
    print(f"  Features: {X.shape[1]} (from {total_neurons} neurons × {N_TIME_BINS if USE_TEMPORAL_BINS else 1} time bins)")
    print(f"  Trials: {X.shape[0]}")
    
    # Train model
    best_pipeline, best_params = train_optimized_logistic(X, y, N_FOLDS)
    
    # Evaluate
    fold_accs = evaluate_model(best_pipeline, X, y, N_FOLDS)
    
    elapsed = time.time() - start_time
    
    # Results
    print(f"\n{'='*70}")
    print("RESULTS")
    print(f"{'='*70}")
    print(f"Accuracy: {np.mean(fold_accs)*100:.2f}% +/- {np.std(fold_accs)*100:.2f}%")
    print(f"Per-fold: {[f'{acc*100:.1f}%' for acc in fold_accs]}")
    print(f"\nBest parameters:")
    for param, value in best_params.items():
        print(f"  {param.replace('classifier__', '')}: {value}")
    print(f"\nTime: {elapsed:.1f}s")
    
    return {
        'file_name': file_name,
        'accuracy_mean': np.mean(fold_accs),
        'accuracy_std': np.std(fold_accs),
        'fold_accuracies': fold_accs,
        'best_params': best_params,
        'n_features': X.shape[1],
        'n_trials': X.shape[0],
        'time_seconds': elapsed
    }


def run_pipeline():
    """Main pipeline."""
    print("="*70)
    print("OPTIMIZED MULTI-PLANE DECODER")
    print("="*70)
    print(f"Target stimuli: {TARGET_STIMULI}")
    print(f"Neurons per plane: {N_TOP_NEURONS_PER_PLANE}")
    print(f"Temporal binning: {USE_TEMPORAL_BINS} ({N_TIME_BINS} bins)")
    print(f"Smoothing: {SMOOTH_TRACES} (sigma={SMOOTH_SIGMA})")
    print()
    
    # Get files
    assets = get_all_nwb_assets(DANDISET_ID)
    
    if MAX_FILES:
        assets = assets[:MAX_FILES]
    
    print(f"Processing {len(assets)} files\n")
    
    all_results = []
    
    for file_idx, asset in enumerate(assets, 1):
        print(f"\n{'='*70}")
        print(f"FILE {file_idx}/{len(assets)}")
        print(f"{'='*70}")
        
        try:
            nwb = load_nwb_file(asset)
            
            result = analyze_file(nwb, asset.path)
            all_results.append(result)
            
        except Exception as e:
            print(f"ERROR: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    # Summary
    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    
    if all_results:
        accs = [r['accuracy_mean'] for r in all_results]
        print(f"\nOverall: {np.mean(accs)*100:.2f}% +/- {np.std(accs)*100:.2f}%")
        print(f"Best: {np.max(accs)*100:.2f}%")
        print(f"Range: [{np.min(accs)*100:.2f}% - {np.max(accs)*100:.2f}%]")
        
        print("\nPer file:")
        for result in all_results:
            file_short = result['file_name'].split('/')[-1][:50]
            print(f"  {file_short}: {result['accuracy_mean']*100:.2f}%")
    
    return all_results


# ============================================================================
# RUN
# ============================================================================

if __name__ == "__main__":
    results = run_pipeline()

OPTIMIZED MULTI-PLANE DECODER
Target stimuli: ['Cinematic', 'Rendered', 'sports1m']
Neurons per plane: 250
Temporal binning: True (3 bins)
Smoothing: True (sigma=2)

Processing 3 files


FILE 1/3

File: sub-17797_ses-4-scan-7_behavior+image+ophys.nwb
  Selecting top 250 neurons per plane...
    RoiResponseSeries1: 250/643 neurons
    RoiResponseSeries2: 250/452 neurons
    RoiResponseSeries3: 250/1455 neurons
    RoiResponseSeries4: 250/1389 neurons
    RoiResponseSeries5: 250/1420 neurons
    RoiResponseSeries6: 250/1411 neurons
    RoiResponseSeries7: 250/895 neurons
    RoiResponseSeries8: 250/730 neurons
  Using temporal binning: 3 bins per trial
  Smoothing traces: sigma=2

Dataset shape:
  Features: 6000 (from 2000 neurons × 3 time bins)
  Trials: 384

RESULTS
Accuracy: 82.26% +/- 6.31%
Per-fold: ['85.7%', '80.5%', '89.6%', '84.4%', '71.1%']

Best parameters:
  C: 0.1
  class_weight: None
  max_iter: 5000
  penalty: l2
  solver: saga

Time: 1080.3s

FILE 2/3

File: sub-17797_ses-